GENERAR DATASET USANDO PANDAS

In [61]:
import pandas as pd
import kagglehub

path = kagglehub.dataset_download("shayanzk/imdb-top-100-movies-dataset-2025-edition")
file = path + "/top_100_movies_full_best_effort.csv"

df = pd.read_csv(file)
df.drop(columns=["Rank","Title","Main Actor(s)", "Director","Country","Language","Box Office ($M)"], inplace=True)
df.head()


,Year,Genre(s),IMDb Rating,Rotten Tomatoes %,Runtime (mins),Oscars Won,Metacritic Score
0,1994,Drama,9.3,91.0,142.0,0,82.0
1,1972,Crime|Drama,9.2,98.0,175.0,3,100.0
2,2008,Action|Crime|Drama,9.0,94.0,152.0,2,84.0
3,1974,Crime|Drama,9.0,97.0,202.0,6,90.0
4,1957,Crime|Drama,9.0,100.0,96.0,0,96.0


ENCODING DE CADENAS DE TEXTO 

In [62]:
# One-hot encoding de las columnas "Genre(s)", "Director", "Language" y "Country"
dummies = pd.concat([df[col].str.get_dummies(sep='|') for col in ['Genre(s)']], axis=1)
df = df.join(dummies).drop(columns=['Genre(s)'])


CONVERTIR A MATRIZ NUMPY 

In [63]:
import numpy as np

X = df.to_numpy()


In [24]:
df.head()

,Year,IMDb Rating,Rotten Tomatoes %,Runtime (mins),Oscars Won,Box Office ($M),Metacritic Score,Action,Adventure,Animation,...,Horror,Music,Mystery,Romance,Satire,Sci-Fi,Sport,Thriller,War,Western
0,1994,9.3,91.0,142.0,0,58.0,82.0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
1,1972,9.2,98.0,175.0,3,246.1,100.0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
2,2008,9.0,94.0,152.0,2,1004.9,84.0,1,0,0,...,0,0,0,0,0,0,0,0,0,0
3,1974,9.0,97.0,202.0,6,48.5,90.0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
4,1957,9.0,100.0,96.0,0,1.0,96.0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


In [25]:
print(X)

[[1994.     9.3   91.  ...    0.     0.     0. ]
 [1972.     9.2   98.  ...    0.     0.     0. ]
 [2008.     9.    94.  ...    0.     0.     0. ]
 ...
 [1944.     8.    98.  ...    0.     0.     0. ]
 [1977.     8.    97.  ...    0.     0.     0. ]
 [1959.     8.2   94.  ...    0.     0.     0. ]]


In [26]:
#Muestra todos los generos que existen en el dataset
genre_columns = [col for col in df.columns if df[col].isin([0,1]).all()]

print(genre_columns)

['Action', 'Adventure', 'Animation', 'Biography', 'Comedy', 'Crime', 'Drama', 'Family', 'Fantasy', 'Film-Noir', 'History', 'Horror', 'Music', 'Mystery', 'Romance', 'Satire', 'Sci-Fi', 'Sport', 'Thriller', 'War', 'Western']


In [27]:
print(df)

    Year  IMDb Rating  Rotten Tomatoes %  Runtime (mins)  Oscars Won  \
0   1994          9.3               91.0           142.0           0   
1   1972          9.2               98.0           175.0           3   
2   2008          9.0               94.0           152.0           2   
3   1974          9.0               97.0           202.0           6   
4   1957          9.0              100.0            96.0           0   
..   ...          ...                ...             ...         ...   
95  1962          8.3               98.0           222.0           7   
96  1957          8.1               93.0           161.0           7   
97  1944          8.0               98.0           107.0           0   
98  1977          8.0               97.0            93.0           4   
99  1959          8.2               94.0           121.0           1   

    Box Office ($M)  Metacritic Score  Action  Adventure  Animation  ...  \
0              58.0              82.0       0          0   

Divide a los datos en clases

In [77]:
#Si el numero de oscar es mayor a 0, entra en la categoria de "ganó oscar" (1), sino, "no ganó oscar" (0)
Y = (df["Oscars Won"] > 0).astype(int).drop(columns=["Oscars Won"])


CLASIFICAR EN UN ARBOL DE DECISIÓN

In [78]:
from sklearn import tree

clf = tree.DecisionTreeClassifier()
clf = clf.fit(X, Y)

PREDICE SI UNA PELICULA GANA EL OSCAR EN FUNCIÓN DE LOS VALORES QUE INGRESEMOS

In [87]:
#La pelicula pertenece a los generos Crimen, Drama,Film-Noir,Mistery,Thriller (Mullholland Drive, 2001)
clf.predict([[
    2001, #Year
    7.9, #IMDb Rating
    83.0, #Rotten Tomatoes %
    147, #Runtime (mins)
    87, #Metacritic Score   
    0, #Animation
    0, #Biography
    0, #Biography
    0, #Comedy
    1, #Crime,
    0, #Drama
    0, #Family
    0, #Fantasy 
    1, #Film-Noir
    0, #History
    0, #Horror
    0, #Music
    0, #Mistery
    0, #Romance
    0, #Satire
    0, #Sport
    0, #Sci-Fi
    0, #Sport
    1, #Thriller
    0, #War
    0 #Western
    ]])

array([1])

In [88]:
#La pelicula pertenece a los generos Acción, Adventure y Sci-Fi (Spider-man, 2002)
clf.predict([[
    2002,  # Year
    7.4,   # IMDb Rating
    90,    # Rotten Tomatoes %
    121,   # Runtime (mins)
    73,    # Metacritic Score
    1,     # Action
    1,     # Adventure
    0,     # Animation
    0,     # Biography
    0,     # Comedy
    0,     # Crime
    0,     # Drama
    0,     # Family
    0,     # Fantasy
    0,     # Film-Noir
    0,     # History
    0,     # Horror
    0,     # Music
    0,     # Mystery
    0,     # Romance
    0,     # Satire
    1,     # Sci-Fi
    0,     # Sport
    0,     # Thriller
    0,     # War
    0      # Western
]])

array([0])

In [89]:
#La pelicula pertenece a los generos Drama,Music,Thriller (Emilia Perez, 2024)
clf.predict([[
    2024,  # Year
    5.3,   # IMDb Rating
    70,    # Rotten Tomatoes %
    130,   # Runtime (mins)
    70,    # Metacritic Score
    0,     # Action
    0,     # Adventure
    0,     # Animation
    0,     # Biography
    0,     # Comedy
    0,     # Crime
    1,     # Drama
    0,     # Family
    0,     # Fantasy
    0,     # Film-Noir
    0,     # History
    0,     # Horror
    1,     # Music
    0,     # Mystery
    0,     # Romance
    0,     # Satire
    0,     # Sci-Fi
    0,     # Sport
    1,     # Thriller
    0,     # War
    0      # Western
]])

array([1])

Hace predicciones solo teniendo en cuenta a los géneros de la pelicula, ignorando otros campos

In [ ]:

X_genres = df[genre_columns].to_numpy()

# Create Y (target variable)
Y = (df["Oscars Won"] > 0).astype(int)

# Train the classifier
clf = tree.DecisionTreeClassifier()
clf = clf.fit(X_genres, Y)

#Mulholland Drive, 2001
clf.predict([[
    0, #Animation
    0, #Biography
    0, #Comedy
    1, #Crime
    0, #Drama
    0, #Family
    0, #Fantasy 
    1, #Film-Noir
    0, #History
    0, #Horror
    0, #Music
    0, #Mystery
    0, #Romance
    0, #Satire
    0, #Sport
    0, #Sci-Fi
    0, #Sport
    1, #Thriller
    0, #War
    0, #Western
    0  #Action
]])



array([0])

In [ ]:

#Spider-man, 2002
clf.predict([[
    1,     # Action
    1,     # Adventure
    0,     # Animation
    0,     # Biography
    0,     # Comedy
    0,     # Crime
    0,     # Drama
    0,     # Family
    0,     # Fantasy
    0,     # Film-Noir
    0,     # History
    0,     # Horror
    0,     # Music
    0,     # Mystery
    0,     # Romance
    0,     # Satire
    1,     # Sci-Fi
    0,     # Sport
    0,     # Thriller
    0,     # War
    0      # Western
]])

In [98]:
#Emilia Perez, 2024

clf.predict([[
    0,     # Action
    0,     # Adventure
    0,     # Animation
    0,     # Biography
    0,     # Comedy
    0,     # Crime
    1,     # Drama
    0,     # Family
    0,     # Fantasy
    0,     # Film-Noir
    0,     # History
    0,     # Horror
    1,     # Music
    0,     # Mystery
    0,     # Romance
    0,     # Satire
    0,     # Sci-Fi
    0,     # Sport
    1,     # Thriller
    0,     # War
    0      # Western
]])

array([1])

In [ ]:
#Hace predicciones teniendo en cuenta a las reseñas de la pelicula, sin tener en cuenta los generos